# Project 7 — Parabolic PDE: Heat Equation (FTCS)
**Topic:** Explicit finite-difference solution of the 1-D diffusion equation. Von Neumann stability analysis, the CFL condition, and comparison with Crank-Nicolson.


---
## 1. Set Up

### Physical Motivation
The **1-D heat equation** (diffusion equation):
$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

governs: temperature diffusion (where $u = T$, $\alpha = k/\rho c_p$), concentration diffusion, quantum wavepacket spreading (where $\alpha = i\hbar/2m$), and probability density evolution in a random walk.

### Initial and Boundary Conditions
$$u(x, 0) = \sin(\pi x / L), \quad u(0,t) = u(L,t) = 0$$

The exact solution (first Fourier mode):
$$u^*(x,t) = \sin(\pi x/L)\exp\!\left(-\alpha\frac{\pi^2}{L^2}t\right)$$

The solution decays exponentially in time: heat leaks out through the boundaries. The decay rate $\alpha\pi^2/L^2$ is the **diffusion eigenvalue** for the first mode.

### FTCS Scheme
**Forward in Time, Centred in Space**:
$$u_i^{n+1} = u_i^n + r(u_{i-1}^n - 2u_i^n + u_{i+1}^n), \qquad r = \frac{\alpha\Delta t}{\Delta x^2}$$

This is **explicit** — the new value $u_i^{n+1}$ depends only on known old values. One step costs $\mathcal{O}(N)$ — cheap per step, but the time step is severely constrained by stability.

### ⚠️ Critical Stability Condition — Von Neumann Analysis
The **CFL (Courant-Friedrichs-Lewy) condition** for FTCS:
$$r = \frac{\alpha\Delta t}{\Delta x^2} \leq \frac{1}{2}$$

**Derivation**: substitute $u_j^n = \xi^n e^{ijk\Delta x}$ (Fourier mode). The amplification factor is:
$$\xi = 1 - 2r(1 - \cos(k\Delta x))$$

For stability $|\xi| \leq 1$ for all $k$. The worst case $k\Delta x = \pi$ gives $\xi = 1 - 4r$, requiring $r \leq 1/2$.

**If $r > 1/2$**: oscillations grow without bound — the solution blows up. The code explicitly checks and raises an error. This is not a numerical glitch — it is a **fundamental instability** of the explicit scheme.

### 🔧 Crank-Nicolson (Improvement)
The Crank-Nicolson scheme averages the spatial operator between time levels $n$ and $n+1$:
$$u_i^{n+1} - \frac{r}{2}\delta^2 u_i^{n+1} = u_i^n + \frac{r}{2}\delta^2 u_i^n$$

This is **unconditionally stable** for all $r$ and is **second-order in both space and time** — compared to FTCS which is first-order in time. The cost: a tridiagonal solve at each time step ($\mathcal{O}(N)$ — same asymptotic cost as FTCS but larger constant).


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_heat_equation")
OUT.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `heat_ftcs()` — Line by Line

```python
r = alpha * dt / dx**2
if r > 0.5:
    raise ValueError(f"FTCS unstable: alpha*dt/dx^2 = {r:.6f} > 0.5")
```
**Stability guard**: computes the CFL number $r$ and rejects the computation if it exceeds $1/2$. This is the most important line in the function — without it, the solver will silently produce garbage (exponentially growing oscillations that look like valid output to an inexperienced user).

```python
U = np.empty((t.size, nx), dtype=float)
U[0] = np.sin(np.pi * x / L)
U[0, 0] = 0.0; U[0, -1] = 0.0
```
Pre-allocates the full solution array. Storing all time steps costs $N_x \times N_t$ memory — for $N_x=101$ and $N_t = T/\Delta t \approx 4000$: $\sim 3\,\text{MB}$. For production code, only store the current and previous steps to save memory.

```python
U[n+1, 1:-1] = U[n, 1:-1] + r*(U[n, :-2] - 2.0*U[n, 1:-1] + U[n, 2:])
```
**FTCS update** — vectorised over all interior points. The three-term expression is $\delta^2 u = u_{i-1} - 2u_i + u_{i+1}$, the discrete Laplacian. No Python loop over spatial points — entire spatial step done with one NumPy operation.

```python
U[n+1, 0] = 0.0; U[n+1, -1] = 0.0
```
**Re-apply boundary conditions at every step.** The FTCS update does not touch boundary points (the `1:-1` slice), but explicitly setting them is good practice — it prevents accumulation of floating-point errors that could corrupt the BC over many steps.

### `analytic_first_mode(x, t)`
```python
return np.sin(np.pi*X/L) * np.exp(-alpha*(np.pi/L)**2 * T)
```
The exact solution for the given IC. Used for quantitative error measurement. The `np.meshgrid` produces 2-D arrays compatible with the full solution `U`.

### ⚠️ Known Weaknesses
1. **First-order in time**: FTCS has local error $\mathcal{O}(\Delta t)$ in time and $\mathcal{O}(\Delta x^2)$ in space. The time error dominates for large $\Delta t$.
2. **Stability constraint**: $\Delta t \leq \Delta x^2/(2\alpha)$. For a 10× finer spatial grid, the time step must be $100\times$ smaller — $1000\times$ more work.
3. **No adaptivity**: fixed $\Delta t$ throughout. If $u$ is smooth (late times), a larger $\Delta t$ would be safe and faster.
4. **Memory**: storing all $N_t$ time steps uses $\mathcal{O}(N_x N_t)$ memory — often unnecessary if only a few time slices are needed.


In [ ]:
def heat_ftcs(alpha=1.0, L=1.0, nx=101, dt=2.0e-5, t_max=0.08):
    x  = np.linspace(0.0, L, nx)
    dx = x[1] - x[0]
    r  = alpha * dt / dx**2        # CFL number: must be <= 0.5 for stability
    if r > 0.5:
        raise ValueError(f"FTCS unstable: r = alpha*dt/dx^2 = {r:.6f} > 0.5")

    t = np.arange(0.0, t_max + 0.5*dt, dt)
    U = np.empty((t.size, nx), dtype=float)
    U[0] = np.sin(np.pi * x / L)  # IC: first Fourier mode
    U[0, 0] = 0.0; U[0, -1] = 0.0

    for n in range(t.size - 1):
        U[n+1]       = U[n]
        U[n+1, 1:-1] = U[n, 1:-1] + r*(U[n, :-2] - 2.0*U[n, 1:-1] + U[n, 2:])
        U[n+1, 0]   = 0.0         # re-enforce left BC
        U[n+1, -1]  = 0.0         # re-enforce right BC

    return x, t, U, r

def analytic_first_mode(x, t, alpha=1.0, L=1.0):
    X, T = np.meshgrid(x, t)
    return np.sin(np.pi*X/L) * np.exp(-alpha*(np.pi/L)**2 * T)

def sample_time_indices(t, n_samples=6):
    return np.linspace(0, len(t)-1, n_samples, dtype=int)


---
## 3 & 4. Calculation & Writeouts

### CFL Number Interpretation
- $r = 0.5$: marginally stable — the Fourier mode $k\Delta x = \pi$ is exactly frozen (amplitude factor $\xi = -1$, oscillates in sign but does not grow).
- $r < 0.5$: stable — all modes decay.
- $r > 0.5$: **unstable** — high-frequency modes grow exponentially.

The code uses $r = 2\times10^{-5} / (10^{-2})^2 = 0.2$ — comfortably below $0.5$.

### What to Observe
- Solution decays exponentially: $u(x,t) \approx e^{-\alpha\pi^2 t}$ times the initial shape.
- At $t = T_{\text{diff}} = L^2/(\alpha\pi^2) \approx 0.1$, the amplitude drops to $1/e$ of its initial value.
- Numerical solution should agree with analytic to within $\mathcal{O}(\Delta t)$ accuracy.


In [ ]:
ALPHA = 1.0; L = 1.0; NX = 101; DT = 2.0e-5; T_MAX = 0.08

x, t, U, r = heat_ftcs(alpha=ALPHA, L=L, nx=NX, dt=DT, t_max=T_MAX)
U_exact     = analytic_first_mode(x, t, alpha=ALPHA, L=L)
max_error   = np.max(np.abs(U - U_exact))

print(f"CFL number r = alpha*dt/dx^2 = {r:.6f}  (must be < 0.5)")
print(f"Grid: nx={NX}, nt={len(t)},  dt={DT:.2e}")
print(f"Max error vs. analytic: {max_error:.6f}")
print(f"Diffusion time T_diff = L^2/(alpha*pi^2) = {L**2/(ALPHA*np.pi**2):.4f}")

# Time slices
sample_idx = sample_time_indices(t, n_samples=6)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(sample_idx)))

ax = axes[0]
for idx, c in zip(sample_idx, colors):
    ax.plot(x, U[idx], color=c, lw=2, label=f't={t[idx]:.4f}')
    ax.plot(x, U_exact[idx], '--', color=c, lw=1.0, alpha=0.6)
ax.set_xlabel('x'); ax.set_ylabel('u(x,t)')
ax.set_title('FTCS solution (solid) vs. analytic (dashed)')
ax.legend(fontsize=8)

# Error evolution
ax2 = axes[1]
errors = np.max(np.abs(U - U_exact), axis=1)
ax2.semilogy(t, errors, 'steelblue', lw=1.5)
ax2.set_xlabel('t'); ax2.set_ylabel('Max pointwise error')
ax2.set_title('Error grows linearly with t (O(dt) temporal accuracy)')
ax2.grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT / "heat_equation.png", dpi=150, bbox_inches='tight')
plt.show()

# Instability demonstration
print("\nDemonstrating instability for r > 0.5:")
try:
    x2, t2, U2, r2 = heat_ftcs(alpha=1.0, L=1.0, nx=101, dt=6.0e-5, t_max=0.001)
    print("  No error raised — check stability guard!")
except ValueError as e:
    print(f"  Correctly caught: {e}")

print("\n" + "=" * 55)
print("SANITY CHECKS — Project 07 Heat Equation")
print("=" * 55)
ok1 = r < 0.5
print(f"  1. CFL stable: r={r:.4f} < 0.5  {'✓' if ok1 else '✗'}")
ok2 = max_error < 0.01
print(f"  2. Max error < 0.01: {max_error:.6f}  {'✓' if ok2 else '✗'}")
ok3 = np.all(np.abs(U[:, 0]) < 1e-12) and np.all(np.abs(U[:, -1]) < 1e-12)
print(f"  3. BCs u(0,t)=u(L,t)=0 throughout:  {'✓' if ok3 else '✗'}")
ok4 = np.all(U[:, 1:-1] >= -1e-10)   # solution should remain non-negative for sin IC
print(f"  4. Solution non-negative (physical):  {'✓' if ok4 else '✗'}")
ok5 = np.max(np.abs(U[0] - np.sin(np.pi*x/L))) < 1e-10
print(f"  5. IC exactly preserved at t=0:  {'✓' if ok5 else '✗'}")
# Check decay rate
log_amp = np.log(np.max(U[:, 1:-1], axis=1))
log_amp = log_amp[np.isfinite(log_amp)]
idx_fit = len(log_amp)//4
coeff   = np.polyfit(t[:len(log_amp)][idx_fit:], log_amp[idx_fit:], 1)
decay_num    = -coeff[0]
decay_theory = ALPHA * (np.pi/L)**2
ok6 = abs(decay_num - decay_theory)/decay_theory < 0.01
print(f"  6. Decay rate: num={decay_num:.4f}, theory={decay_theory:.4f}  "
      f"err={abs(decay_num-decay_theory)/decay_theory*100:.2f}%  {'✓' if ok6 else '✗'}")
print("=" * 55)
